#  NESP - ThermoNet v2: Exploration & Visualisation des voxel features

Ce notebook fait 4 choses:
Ce projet propose une approche hybride avancée pour prédire la stabilité des enzymes et la température de fusion ($T_m$) suite à des mutations d'acides aminés dans le cadre de compétitions de bio-informatique. Le pipeline combine deux volets complémentaires : d'une part, l'apprentissage profond via un réseau de neurones convolutifs 3D (3D CNN) sous PyTorch appliqué à des matrices de voxels 3D (capturant les modifications atomiques, les propriétés hydrophobes, les liaisons hydrogène et les charges) pour modéliser les variations de stabilité ($ddG$ et $dT$) ; d'autre part, un modèle d'apprentissage automatique tabulaire basé sur XGBoost avec une validation croisée rigoureuse pour optimiser et prédire avec précision la température de fusion ($T_m$) à partir des caractéristiques de mutations.

**Sources de donnees (Kaggle inputs a ajouter via "+ Add Input"):**
- Competition officielle: `novozymes-enzyme-stability-prediction`
- Notebook output: `vslaykovsky/14656-unique-mutations-voxel-features-pdbs`


## 1) Imports
**Cette partie contient les bibliothèques nécessaires (Plotly, Pandas, NumPy). Elle inclut également la configuration pour que Plotly fonctionne correctement (via 'iframe') et évite d'afficher des graphiques vides.**

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from tqdm.notebook import tqdm

pio.renderers.default = 'iframe'
print('Renderer plotly actif:', pio.renderers.default)

import plotly.express as _px_test


Renderer plotly actif: iframe


## 2) Chargement des datasets

**Ici, nous configurons les chemins d'accès aux données selon que vous êtes sur Kaggle ou sur votre PC (en local). Le code lit le fichier CSV, rassemble les chemins des fichiers .npy (voxels) et filtre uniquement les mutations dont les fichiers existent.**


In [ ]:
RUNNING_ON_KAGGLE = os.path.exists('/kaggle/input')
print('Environnement detecte:', 'Kaggle' if RUNNING_ON_KAGGLE else 'Local (PC)')

if RUNNING_ON_KAGGLE:
    COMPETITION_DIR = '../input/competitions/novozymes-enzyme-stability-prediction'
    VOXEL_DATASET_DIR = '../input/notebooks/vslaykovsky/14656-unique-mutations-voxel-features-pdbs'
else:
    COMPETITION_DIR = './data/novozymes-enzyme-stability-prediction'
    VOXEL_DATASET_DIR = './data/14656-unique-mutations-voxel-features-pdbs'

WILDTYPE_PDB = os.path.join(COMPETITION_DIR, 'wildtype_structure_prediction_af2.pdb')
TEST_CSV = os.path.join(COMPETITION_DIR, 'test.csv')
CSV_PATH = os.path.join(VOXEL_DATASET_DIR, 'dataset.csv')
FEATURES_DIR = os.path.join(VOXEL_DATASET_DIR, 'features')

if RUNNING_ON_KAGGLE:
    for root, dirs, files in os.walk('../input'):
        depth = root.count(os.sep) - '../input'.count(os.sep)
        if depth > 3:
            dirs[:] = []
            continue

df = pd.read_csv(CSV_PATH)

df['features_path'] = df.apply(
    lambda r: os.path.join(
        FEATURES_DIR, f'{r.PDB_chain}_{r.wildtype}{r.pdb_position}{r.mutant}.npy'
    ),
    axis=1,
)

df['features_exists'] = df['features_path'].apply(os.path.exists)
df = df[df['features_exists']].reset_index(drop=True)
df.head()

Environnement detecte: Kaggle
COMPETITION_DIR  : ../input/competitions/novozymes-enzyme-stability-prediction
VOXEL_DATASET_DIR: ../input/notebooks/vslaykovsky/14656-unique-mutations-voxel-features-pdbs


## 3) Visualisation de data 14656-unique-mutations-voxel-features-pdbs
**(3d voxel , ddG, dT, ddG vs dT , 14features (wildtype vs mutant) )**


In [ ]:
print(f'1. Chargement du fichier CSV: {CSV_PATH}')
df = pd.read_csv(CSV_PATH)
print(f'   Total mutations dans dataset.csv: {len(df)}')

df['features_path'] = df.apply(
    lambda r: os.path.join(
        FEATURES_DIR, f'{r.PDB_chain}_{r.wildtype}{r.pdb_position}{r.mutant}.npy'
    ),
    axis=1,
)

df['features_exists'] = df['features_path'].apply(os.path.exists)
n_before = len(df)
df = df[df['features_exists']].reset_index(drop=True)
print(f'   Mutations avec features .npy disponibles: {len(df)} / {n_before}')

df.head()


1. Chargement du fichier CSV: ../input/notebooks/vslaykovsky/14656-unique-mutations-voxel-features-pdbs/dataset.csv
   Total mutations dans dataset.csv: 14656
   Mutations avec features .npy disponibles: 12167 / 14656


,sequence,wildtype,pdb_position,seq_position,mutant,ddG,dT,wT,pH,source,PDB_chain,features_path,features_exists
0,AAQASVVANQLIPINTALTLVMMRSEVVTPVGIPAEDIPRLVSMQV...,D,36,36,A,0.705833,NaN,NaN,7.0,"['Q3421.txt', 'Q3214_direct.csv', 'Q1744_direc...",1msiA,../input/notebooks/vslaykovsky/14656-unique-mu...,True
1,AAQASVVANQLIPINTALTLVMMRSEVVTPVGIPAEDIPRLVSMQV...,D,58,58,N,-0.120000,NaN,NaN,7.0,"['Q3421.txt', 'Q3214_direct.csv', 'Q1744_direc...",1msiA,../input/notebooks/vslaykovsky/14656-unique-mu...,True
2,AAQASVVANQLIPINTALTLVMMRSEVVTPVGIPAEDIPRLVSMQV...,E,25,25,A,-0.050000,NaN,NaN,7.0,"['Q3421.txt', 'Q3214_direct.csv', 'Q1744_direc...",1msiA,../input/notebooks/vslaykovsky/14656-unique-mu...,True
3,AAQASVVANQLIPINTALTLVMMRSEVVTPVGIPAEDIPRLVSMQV...,R,23,23,A,-0.763333,NaN,NaN,7.0,"['Q3421.txt', 'Q3214_direct.csv', 'Q1744_direc...",1msiA,../input/notebooks/vslaykovsky/14656-unique-mu...,True
4,AAQASVVANQLIPINTALTLVMMRSEVVTPVGIPAEDIPRLVSMQV...,R,39,39,A,-0.726667,NaN,NaN,7.0,"['Q3421.txt', 'Q3214_direct.csv', 'Q1744_direc...",1msiA,../input/notebooks/vslaykovsky/14656-unique-mu...,True


## 3) Visualisation de ddG, dT et ddG vs dT


In [ ]:

fig_ddg = px.histogram(
    df, x='ddG', nbins=60,
    title="Distribution de \u0394\u0394G (ddG) - changement d'energie de stabilite",
)
fig_ddg.update_layout(bargap=0.02)
fig_ddg.show()


In [ ]:

fig_dt = px.histogram(
    df, x='dT', nbins=60,
    title='Distribution de \u0394T (dT) - changement de temperature de fusion (Tm)',
)
fig_dt.update_layout(bargap=0.02)
fig_dt.show()


In [ ]:

fig_scatter = px.scatter(
    df, x='ddG', y='dT',
    title='ddG vs dT',
    opacity=0.4,
    trendline='ols',  
)
fig_scatter.show()


## 4) Visualisation 3D des voxels

Chaque fichier `.npy` a la forme `(14, 16, 16, 16)`:
- canaux 0-6  = 7 features du **wildtype**
- canaux 7-13 = 7 features du **mutant** (meme ordre de features)

Ordre des 7 features (standard ThermoNet / HTMD `getVoxelDescriptors`): `hydrophobic, aromatic, hbond_acceptor, hbond_donor, positive_ionizable, negative_ionizable, occupancies`


In [ ]:
FEATURE_NAMES = [
    'hydrophobic', 'aromatic', 'hbond_acceptor', 'hbond_donor',
    'positive_ionizable', 'negative_ionizable', 'occupancies',
]

VOXEL_SIZE = 16
GRID_COORDS = np.array(
    [(x, y, z) for x in range(VOXEL_SIZE) for y in range(VOXEL_SIZE) for z in range(VOXEL_SIZE)]
)

def plot_feature_diff(sample_idx: int, feature: str = 'occupancies',
                       threshold: float = 0.5, marker_size: int = 6):
    row = df.iloc[sample_idx]
    features = np.load(row.features_path)

    ch = FEATURE_NAMES.index(feature)
    wildtype_vals = features[ch].flatten()
    mutant_vals = features[7 + ch].flatten()

    mask_wt = wildtype_vals > threshold
    mask_mut = mutant_vals > threshold

    both_mask = mask_wt & mask_mut
    mutant_only_mask = mask_mut & ~mask_wt
    wildtype_only_mask = mask_wt & ~mask_mut

    categories = [
        (both_mask, 'blue', 'blue'),
        (mutant_only_mask, 'red', 'red'),
        (wildtype_only_mask, 'green', 'green'),
    ]

    fig = go.Figure()
    for mask, color, label in categories:
        fig.add_trace(
            go.Scatter3d(
                x=GRID_COORDS[mask, 0], y=GRID_COORDS[mask, 1], z=GRID_COORDS[mask, 2],
                mode='markers', marker=dict(size=marker_size, color=color, opacity=0.85),
                name=label, legendgroup='color',
            )
        )

    ddg_txt = f'{row.ddG:.2f}' if pd.notna(row.ddG) else 'NA'
    fig.update_layout(
        title=f'Train idx:{sample_idx}; ddg={ddg_txt}',
        height=750, legend_title_text='color',
        scene=dict(xaxis_title='x (voxel)', yaxis_title='y (voxel)', zaxis_title='z (voxel)'),
    )
    fig.show(config={'displayModeBar': True, 'scrollZoom': True})

for i in range(min(3, len(df))):
    plot_feature_diff(i, feature='occupancies', threshold=0.5)
    plot_feature_diff(i , feature = 'occupancies', threshold = 0.3)
    

## 5) Comparaison multi-features (occupancies / hydrophobic / hbond_donor)

Meme logique 3 couleurs (bleu = present dans les 2, rouge = mutant seulement,
vert = wildtype seulement), mais affichee **cote a cote** pour plusieurs
features en meme temps, sur un seul sample.


In [ ]:
def plot_multi_feature_diff(sample_idx: int,
                             features=('occupancies', 'hydrophobic', 'hbond_donor'),
                             threshold: float = 0.5, marker_size: int = 5):

    row = df.iloc[sample_idx]
    arr = np.load(row.features_path)  
    n = len(features)
    fig = make_subplots(
        rows=1, cols=n,
        specs=[[{'type': 'scatter3d'}] * n],
        subplot_titles=list(features),
    )

    colors = {'both': 'blue', 'mutant_only': 'red', 'wildtype_only': 'green'}

    for col, feat_name in enumerate(features, start=1):
        ch = FEATURE_NAMES.index(feat_name)
        wt_vals = arr[ch].flatten()
        mut_vals = arr[7 + ch].flatten()

        mask_wt = wt_vals > threshold
        mask_mut = mut_vals > threshold

        both_mask = mask_wt & mask_mut
        mutant_only_mask = mask_mut & ~mask_wt
        wildtype_only_mask = mask_wt & ~mask_mut

        for mask, key in [(both_mask, 'both'),
                           (mutant_only_mask, 'mutant_only'),
                           (wildtype_only_mask, 'wildtype_only')]:
            fig.add_trace(
                go.Scatter3d(
                    x=GRID_COORDS[mask, 0],
                    y=GRID_COORDS[mask, 1],
                    z=GRID_COORDS[mask, 2],
                    mode='markers',
                    marker=dict(size=marker_size, color=colors[key], opacity=0.85),
                    name=key,
                    legendgroup=key,
                    showlegend=(col == 1), 
                ),
                row=1, col=col,
            )

    ddg_txt = f'{row.ddG:.2f}' if pd.notna(row.ddG) else 'NA'
    fig.update_layout(
        title=f'Train idx:{sample_idx}; ddg={ddg_txt} | threshold={threshold}',
        height=600,
        width=380 * n,
        legend_title_text='color',
    )
    fig.show(config={'displayModeBar': True, 'scrollZoom': True})



plot_multi_feature_diff(0, features=('occupancies', 'hydrophobic', 'hbond_donor'), threshold=0.5)


## 6) Statistiques : combien de voxels colores, et combien de samples disponibles ?
**Le principe de coloration de voxel de chaque sample :**
haque mutation est représentée par une matrice de voxels 3D contenant 14 canaux au total (7 pour la protéine d'origine ou wildtype, et 7 pour le mutant). Pour analyser visuellement les changements structuraux, le système compare les deux états à l'aide d'un code couleur intuitif : les voxels inchangés et communs aux deux structures s'affichent en bleu, ceux qui étaient présents uniquement dans l'original et qui ont disparu s'affichent en vert, tandis que les nouveaux voxels introduits par la mutation apparaissent en rouge.
Deux questions distinctes :
- **Combien de voxels** (bleu/rouge/vert) apparaissent pour UN sample donne (sur un total de 16x16x16 = 4096 voxels par feature) ?
- **Combien de mutations (train idx)** avons-nous au total dans `df`, pretes a etre visualisees/entrainees (celles dont le fichier `.npy` existe reellement) ?


In [ ]:
def count_diff_voxels(sample_idx: int, feature: str = 'occupancies', threshold: float = 0.5):

    row = df.iloc[sample_idx]
    arr = np.load(row.features_path)

    ch = FEATURE_NAMES.index(feature)
    wt_vals = arr[ch].flatten()
    mut_vals = arr[7 + ch].flatten()

    mask_wt = wt_vals > threshold
    mask_mut = mut_vals > threshold

    both = int((mask_wt & mask_mut).sum())
    mutant_only = int((mask_mut & ~mask_wt).sum())
    wildtype_only = int((mask_wt & ~mask_mut).sum())
    total_voxels = GRID_COORDS.shape[0]  # 4096

    return {
        'sample_idx': sample_idx,
        'feature': feature,
        'ddG': row.ddG,
        'both_blue': both,
        'mutant_only_red': mutant_only,
        'wildtype_only_green': wildtype_only,
        'total_colored': both + mutant_only + wildtype_only,
        'total_voxels': total_voxels,
    }



stats_rows = [count_diff_voxels(i, feature='occupancies', threshold=0.5)
              for i in range(min(5, len(df)))]
stats_df = pd.DataFrame(stats_rows)
print(stats_df)

print()

print(f"Nombre total de mutations dans df (avec fichier .npy existant): {len(df)}")
print(f"Chaque sample a un espace de {GRID_COORDS.shape[0]} voxels possibles (16x16x16), par feature.")


   sample_idx      feature       ddG  both_blue  mutant_only_red  \
0           0  occupancies  0.705833       1545              127   
1           1  occupancies -0.120000        989               46   
2           2  occupancies -0.050000       1107               42   
3           3  occupancies -0.763333       1193               57   
4           4  occupancies -0.726667       1191               52   

   wildtype_only_green  total_colored  total_voxels  
0                  145           1817          4096  
1                   42           1077          4096  
2                   83           1232          4096  
3                  104           1354          4096  
4                  138           1381          4096  

Nombre total de mutations dans df (avec fichier .npy existant): 12167
Chaque sample a un espace de 4096 voxels possibles (16x16x16), par feature.


## 7) Comparaison de mutations avec des ddG differents

On cherche, pour chaque valeur de ddG demandee, la mutation la plus
proche dans `df` (recherche par distance minimale, car les valeurs
exactes en float peuvent varier legerement), puis on affiche les 3
mutations **cote a cote** avec le meme code couleur (bleu/rouge/vert)
pour voir comment la distribution des voxels occupes change selon
l'intensite (et le signe) du ddG.


In [ ]:
def find_closest_idx(target_ddg: float):
   
    diffs = (df['ddG'] - target_ddg).abs()
    return diffs.idxmin()


def plot_ddg_comparison(ddg_values, feature: str = 'occupancies',
                         threshold: float = 0.5, marker_size: int = 5):

   
    sample_indices = [find_closest_idx(v) for v in ddg_values]

    n = len(sample_indices)
    subplot_titles = []
    for target, idx in zip(ddg_values, sample_indices):
        actual_ddg = df.loc[idx, 'ddG']
        subplot_titles.append(f'idx={idx} | ddg={target} | reel={actual_ddg:.4f}')

    fig = make_subplots(
        rows=1, cols=n,
        specs=[[{'type': 'scatter3d'}] * n],
        subplot_titles=subplot_titles,
    )

    colors = {'both': 'blue', 'mutant_only': 'red', 'wildtype_only': 'green'}
    ch = FEATURE_NAMES.index(feature)

    for col, idx in enumerate(sample_indices, start=1):
        row = df.loc[idx]
        arr = np.load(row.features_path)

        wt_vals = arr[ch].flatten()
        mut_vals = arr[7 + ch].flatten()

        mask_wt = wt_vals > threshold
        mask_mut = mut_vals > threshold

        both_mask = mask_wt & mask_mut
        mutant_only_mask = mask_mut & ~mask_wt
        wildtype_only_mask = mask_wt & ~mask_mut

        for mask, key in [(both_mask, 'both'),
                           (mutant_only_mask, 'mutant_only'),
                           (wildtype_only_mask, 'wildtype_only')]:
            fig.add_trace(
                go.Scatter3d(
                    x=GRID_COORDS[mask, 0],
                    y=GRID_COORDS[mask, 1],
                    z=GRID_COORDS[mask, 2],
                    mode='markers',
                    marker=dict(size=marker_size, color=colors[key], opacity=0.85),
                    name=key,
                    legendgroup=key,
                    showlegend=(col == 1),
                ),
                row=1, col=col,
            )

    fig.update_layout(
        title=f'Comparaison ddG (feature={feature}, threshold={threshold})',
        height=600,
        width=420 * n,
        legend_title_text='color',
    )
    fig.show(config={'displayModeBar': True, 'scrollZoom': True})

    summary = []
    for target, idx in zip(ddg_values, sample_indices):
        summary.append(count_diff_voxels(idx, feature=feature, threshold=threshold))
    return pd.DataFrame(summary)



ddg_targets = [0.705833, -0.120000, -0.050000]
summary_df = plot_ddg_comparison(ddg_targets, feature='occupancies', threshold=0.5)
print(summary_df)


   sample_idx      feature       ddG  both_blue  mutant_only_red  \
0           0  occupancies  0.705833       1545              127   
1           1  occupancies -0.120000        989               46   
2         791  occupancies -0.050000       1658               78   

   wildtype_only_green  total_colored  total_voxels  
0                  145           1817          4096  
1                   42           1077          4096  
2                   94           1830          4096  


## 8) Est-ce que ddG est vraiment correle avec la proportion de bleu ?

On teste l'hypothese sur un echantillon large (pas juste 3 exemples) :
on calcule, pour N mutations aleatoires, la fraction de voxels 'bleu'
(inchange) par rapport au total colore, et on regarde sa correlation
avec ddG. Si la correlation est faible/nulle, ca confirme que ddG n'est
PAS directement determine par la quantite de changement physique brut.


In [ ]:
import numpy as np

N_SAMPLES = 500  

sample_idx_list = np.random.RandomState(42).choice(len(df), size=min(N_SAMPLES, len(df)), replace=False)

rows = []
for idx in sample_idx_list:
    stats = count_diff_voxels(idx, feature='occupancies', threshold=0.5)
    total = stats['total_colored']
    blue_fraction = stats['both_blue'] / total if total > 0 else np.nan
    rows.append({'idx': idx, 'ddG': stats['ddG'], 'blue_fraction': blue_fraction,
                 'total_colored': total})

corr_df = pd.DataFrame(rows).dropna()

# Correlation de Pearson entre ddG et blue_fraction
correlation = corr_df['ddG'].corr(corr_df['blue_fraction'])
print(f"Correlation (Pearson) entre ddG et fraction de bleu: {correlation:.3f}")
print(f"(proche de 0 = pas de lien direct ; proche de +-1 = lien fort)")
print()
print(corr_df.describe())

# Visualisation
fig_corr = px.scatter(
    corr_df, x='ddG', y='blue_fraction',
    title=f'ddG vs fraction de voxels inchanges (bleu) | correlation={correlation:.3f}',
    opacity=0.5, trendline='ols',
)
fig_corr.show(config={'displayModeBar': True})


Correlation (Pearson) entre ddG et fraction de bleu: 0.072
(proche de 0 = pas de lien direct ; proche de +-1 = lien fort)

                idx         ddG  blue_fraction  total_colored
count    446.000000  446.000000     446.000000     446.000000
mean    5852.163677   -1.111162       0.801699    1672.147982
std     3551.203450    1.928548       0.146612     534.314674
min       19.000000  -10.300000       0.397590     265.000000
25%     2626.500000   -2.164485       0.764269    1266.500000
50%     5616.000000   -0.619167       0.856569    1644.500000
75%     9091.750000    0.027752       0.903154    2049.500000
max    12092.000000    7.930000       0.967769    3053.000000


**il n'y a effectivement aucune corrélation linéaire entre le $\Delta\Delta G$ et la fraction de voxels inchangés (blue_fraction).
Cela signifie que l'ampleur de l'effet de la mutation sur la stabilité thermodynamique ($\Delta\Delta G$) ne peut pas être prédite simplement par la proportion globale de voxels qui restent identiques.**

## 9) - SPLit KFold 
Ici, nous divisons les données en 5 folds (N_FOLDS = 5) pour effectuer l'entraînement et la validation du modèle. GroupKFold avec le groupe PDB_chain garantit que toutes les mutations d'une même protéine sont regroupées ensemble dans le même fold.

In [ ]:
from sklearn.model_selection import GroupKFold
import numpy as np

N_FOLDS = 5  

gkf = GroupKFold(n_splits=N_FOLDS)
df['fold'] = -1


for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(df, groups=df['PDB_chain'])):
    df.loc[val_idx, 'fold'] = fold_idx

print(df['fold'].value_counts().sort_index())

fold
0    2434
1    2434
2    2433
3    2433
4    2433
Name: count, dtype: int64


## 10) - Dataset PyTorch (chargement + feature différentielle)
**Cette partie définit la classe VoxelDataset qui permet à PyTorch de lire les données. Elle charge les matrices numpy 3D (les voxels) depuis ton disque, les convertit en tenseurs (le format que PyTorch comprend), et les associe à leur valeur cible ddG si on est en phase d'entraînement**


In [ ]:
import torch
import random
from torch.utils.data import Dataset

class VoxelDataset(Dataset):
    def __init__(self, dataframe, ddg_mean=0.0, ddg_std=1.0, dt_mean=0.0, dt_std=1.0,
                 augment=False):
 
        self.df = dataframe.reset_index(drop=True)
        self.ddg_mean, self.ddg_std = ddg_mean, ddg_std
        self.dt_mean, self.dt_std = dt_mean, dt_std
        self.augment = augment

        self.rot_axes_choices = [(1, 2), (1, 3), (2, 3)]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        arr = np.load(row.features_path).astype(np.float32)  # shape (14,16,16,16)
        arr = arr.copy()
        arr[7:] -= arr[:7]

        if self.augment:

            axes = random.choice(self.rot_axes_choices)
            k = random.randint(0, 3)
            if k > 0:
                arr = np.rot90(arr, k=k, axes=axes).copy()  

        x = torch.from_numpy(arr)  # (14, 16, 16, 16)
        y_ddg = torch.tensor((row.ddG - self.ddg_mean) / self.ddg_std, dtype=torch.float32)
        y_dt = torch.tensor((row.dT - self.dt_mean) / self.dt_std, dtype=torch.float32)

        return x, y_ddg, y_dt


## 11) - Définition du Modèle (Réseau de Neurones 3D)
**Ici, on construit l'architecture du réseau. Comme les données sont spatiales (en 3D), on utilise un CNN 3D (Réseau de Neurones Convolutifs 3D). Le réseau prend en entrée les 14 canaux de caractéristiques des voxels (in_channels=14), extrait les caractéristiques spatiales grâce aux couches Conv3d, réduit la dimension avec MaxPool3d, et utilise enfin des couches Linear pour prédire une valeur continue (le ddG).**

In [ ]:
import torch.nn as nn
class ThermoNet3D(nn.Module):
    def __init__(self, in_channels=14):
        super().__init__()
        
        self.conv_block1 = nn.Sequential(
            nn.Conv3d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.MaxPool3d(2)
        )
        
        self.conv_block2 = nn.Sequential(
            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(),
            nn.MaxPool3d(2)
        )
        
        self.conv_block3 = nn.Sequential(
            nn.Conv3d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool3d(1)
        )
        
        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

## 12)- Boucle d'Entraînement et de Validation
**Sépare tes données (Fold 0 pour la validation, les autres pour l'entraînement), et lance l'apprentissage. Il calcule l'erreur avec MSELoss (Erreur Quadratique Moyenne) et optimise les poids avec l'algorithme Adam. Si le modèle s'améliore sur les données de validation, il sauvegarde les meilleurs poids dans un fichier .pth.**

In [ ]:
from torch.utils.data import DataLoader

C_DT = 0.01  

def train_one_fold(fold, df, epochs=20, batch_size=32, lr=1e-4, device='cuda'):
    train_df = df[df['fold'] != fold]
    val_df = df[df['fold'] == fold]

    ddg_mean, ddg_std = train_df['ddG'].mean(), train_df['ddG'].std()
    dt_mean, dt_std = train_df['dT'].mean(), train_df['dT'].std()
    norm_stats = {'ddg_mean': ddg_mean, 'ddg_std': ddg_std,
                  'dt_mean': dt_mean, 'dt_std': dt_std}
    print(f"[Fold {fold}] Normalisation -> ddG: mean={ddg_mean:.3f} std={ddg_std:.3f} | "
          f"dT: mean={dt_mean:.3f} std={dt_std:.3f}")

    train_ds = VoxelDataset(train_df, ddg_mean=ddg_mean, ddg_std=ddg_std,
                             dt_mean=dt_mean, dt_std=dt_std, augment=True)
    val_ds = VoxelDataset(val_df, ddg_mean=ddg_mean, ddg_std=ddg_std,
                           dt_mean=dt_mean, dt_std=dt_std, augment=False)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = ThermoNet3D().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    mse = nn.MSELoss()

    losses = []
    ddg_losses = []
    dt_losses = []

    for epoch in range(epochs):
        model.train()
        epoch_loss, epoch_ddg_loss, epoch_dt_loss = 0.0, 0.0, 0.0

        for x, y_ddg, y_dt in train_loader:
            x, y_ddg, y_dt = x.to(device), y_ddg.to(device), y_dt.to(device)

            optimizer.zero_grad()
            pred_ddg, pred_dt = model(x)

         
            mask_ddg = ~torch.isnan(y_ddg)
            mask_dt = ~torch.isnan(y_dt)

            loss_ddg = mse(pred_ddg[mask_ddg], y_ddg[mask_ddg]) if mask_ddg.any() \
                else torch.tensor(0.0, device=device, requires_grad=True)
            loss_dt = mse(pred_dt[mask_dt], y_dt[mask_dt]) if mask_dt.any() \
                else torch.tensor(0.0, device=device, requires_grad=True)

            loss = loss_ddg + C_DT * loss_dt

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * x.size(0)
            epoch_ddg_loss += loss_ddg.item() * x.size(0)
            epoch_dt_loss += loss_dt.item() * x.size(0)

        n = len(train_df)
        losses.append(epoch_loss / n)
        ddg_losses.append(epoch_ddg_loss / n)
        dt_losses.append(epoch_dt_loss / n)

        print(f"[Fold {fold}] Epoch {epoch+1}/{epochs} - loss={losses[-1]:.4f} "
              f"(ddg={ddg_losses[-1]:.4f}, dt={dt_losses[-1]:.4f})")

    return model, val_loader, norm_stats, {'losses': losses, 'ddg_losses': ddg_losses, 'dt_losses': dt_losses}


## 13)- Fonction d'evaluation (manquante dans la version precedente )


In [ ]:
from scipy.stats import spearmanr

@torch.no_grad()
def evaluate_fold(model, val_loader, norm_stats, device='cuda'):

    model.eval()

    all_pred_ddg, all_true_ddg = [], []
    all_pred_dt, all_true_dt = [], []

    for x, y_ddg, y_dt in val_loader:
        x = x.to(device)
        pred_ddg, pred_dt = model(x)

        all_pred_ddg.append(pred_ddg.cpu())
        all_true_ddg.append(y_ddg)
        all_pred_dt.append(pred_dt.cpu())
        all_true_dt.append(y_dt)

    pred_ddg = torch.cat(all_pred_ddg).numpy()
    true_ddg = torch.cat(all_true_ddg).numpy()
    pred_dt = torch.cat(all_pred_dt).numpy()
    true_dt = torch.cat(all_true_dt).numpy()


    pred_ddg = pred_ddg * norm_stats['ddg_std'] + norm_stats['ddg_mean']
    true_ddg = true_ddg * norm_stats['ddg_std'] + norm_stats['ddg_mean']
    pred_dt = pred_dt * norm_stats['dt_std'] + norm_stats['dt_mean']
    true_dt = true_dt * norm_stats['dt_std'] + norm_stats['dt_mean']

    mask_ddg = ~np.isnan(true_ddg)
    mask_dt = ~np.isnan(true_dt)

    mse_ddg = float(np.mean((pred_ddg[mask_ddg] - true_ddg[mask_ddg]) ** 2)) if mask_ddg.any() else float('nan')
    mse_dt = float(np.mean((pred_dt[mask_dt] - true_dt[mask_dt]) ** 2)) if mask_dt.any() else float('nan')

    corr_ddg = float(spearmanr(pred_ddg[mask_ddg], true_ddg[mask_ddg]).correlation) if mask_ddg.sum() > 1 else float('nan')
    corr_dt = float(spearmanr(pred_dt[mask_dt], true_dt[mask_dt]).correlation) if mask_dt.sum() > 1 else float('nan')

    print(f"  -> val MSE ddg={mse_ddg:.4f} | val MSE dt={mse_dt:.4f} | "
          f"Spearman ddg={corr_ddg:.4f} | Spearman dt={corr_dt:.4f}")

    return {'mse_ddg': mse_ddg, 'mse_dt': mse_dt, 'corr_ddg': corr_ddg, 'corr_dt': corr_dt}


**Boucle complète sur tous les folds**

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

all_results = []

for fold in range(N_FOLDS):
    model, val_loader, norm_stats, train_history = train_one_fold(fold, df, epochs=20, device=DEVICE)
    eval_result = evaluate_fold(model, val_loader, norm_stats, device=DEVICE)
    all_results.append({**train_history, **eval_result})

mean_spearman_ddg = np.mean([r['corr_ddg'] for r in all_results])
print(f"\nSpearman ddG moyen sur {N_FOLDS} folds: {mean_spearman_ddg:.4f}")
